# Day 37 – Q2: TF-IDF Hand Calculation & Verification

Manual computation of TF, IDF, TF-IDF for 'fabric' in Clothing reviews (Doc_42), verified with code.

In [4]:
import numpy as np
import pandas as pd
import re
from collections import Counter
import math

REVIEWS_PATH = "shopsense_reviews.csv"
TARGET_WORD = "fabric"
STOP_WORD = "the"
RARE_WORD = "embroidery"
DOC_INDEX = 42


In [5]:
def load_clothing_reviews(path):
    try:
        df = pd.read_csv(path)
        assert "review_text" in df.columns and "category" in df.columns
        clothing = df[df["category"] == "Clothing"].reset_index(drop=True)
        return df, clothing
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {path}")

df_all, df_clothing = load_clothing_reviews(REVIEWS_PATH)
print("Total reviews:", len(df_all))
print("Clothing reviews:", len(df_clothing))


Total reviews: 10199
Clothing reviews: 1701


In [6]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return " ".join(text.split())

def tokenize(text):
    return clean_text(text).split()

df_all["tokens"] = df_all["review_text"].apply(tokenize)
df_clothing["tokens"] = df_clothing["review_text"].apply(tokenize)

doc42_tokens = df_clothing["tokens"].iloc[DOC_INDEX]
doc42_text   = df_clothing["review_text"].iloc[DOC_INDEX]
print(f"Doc_42 original text: {doc42_text}")
print(f"Doc_42 tokens ({len(doc42_tokens)} words): {doc42_tokens}")


Doc_42 original text: Product bahut accha hai. Quality is top notch. Will buy again for sure.
Doc_42 tokens (13 words): ['product', 'bahut', 'accha', 'hai', 'quality', 'is', 'top', 'notch', 'will', 'buy', 'again', 'for', 'sure']


## (a) Compute TF, IDF, TF-IDF for 'fabric' in Doc_42 — Step by Step

In [7]:
def compute_tf_word(word, tokens):
    count_word = tokens.count(word)
    total_words = len(tokens) if tokens else 1
    return count_word, total_words, count_word / total_words

def compute_idf_word(word, token_series, N):
    df_count = sum(1 for tokens in token_series if word in tokens)
    idf = math.log((N + 1) / (df_count + 1)) + 1
    return df_count, idf

N_corpus  = len(df_all)

count_fabric, total_doc42, tf_fabric = compute_tf_word(TARGET_WORD, doc42_tokens)
df_count_fabric, idf_fabric = compute_idf_word(TARGET_WORD, df_all["tokens"], N_corpus)
tfidf_fabric = tf_fabric * idf_fabric

print("=== TF('fabric', Doc_42) ===")
print(f"  Count of 'fabric' in Doc_42  : {count_fabric}")
print(f"  Total tokens in Doc_42       : {total_doc42}")
print(f"  TF = {count_fabric}/{total_doc42} = {tf_fabric:.6f}")
print(f"\n=== IDF('fabric', 10K corpus) ===")
print(f"  N (total docs)               : {N_corpus}")
print(f"  df (docs containing fabric)  : {df_count_fabric}")
print(f"  IDF = log(({N_corpus}+1)/({df_count_fabric}+1))+1 = {idf_fabric:.6f}")
print(f"\n=== TF-IDF('fabric', Doc_42) ===")
print(f"  TF-IDF = {tf_fabric:.6f} * {idf_fabric:.6f} = {tfidf_fabric:.6f}")


=== TF('fabric', Doc_42) ===
  Count of 'fabric' in Doc_42  : 0
  Total tokens in Doc_42       : 13
  TF = 0/13 = 0.000000

=== IDF('fabric', 10K corpus) ===
  N (total docs)               : 10199
  df (docs containing fabric)  : 0
  IDF = log((10199+1)/(0+1))+1 = 10.230143

=== TF-IDF('fabric', Doc_42) ===
  TF-IDF = 0.000000 * 10.230143 = 0.000000


## (b) IDF('the') vs IDF('embroidery')

In [8]:
df_the, idf_the = compute_idf_word(STOP_WORD, df_all["tokens"], N_corpus)
df_emb, idf_emb = compute_idf_word(RARE_WORD, df_all["tokens"], N_corpus)

print(f"IDF('the'):")
print(f"  Docs containing 'the'        : {df_the}/{N_corpus}")
print(f"  IDF = log(({N_corpus}+1)/({df_the}+1))+1 = {idf_the:.6f}")
print(f"\nIDF('embroidery'):")
print(f"  Docs containing 'embroidery' : {df_emb}/{N_corpus}")
print(f"  IDF = log(({N_corpus}+1)/({df_emb}+1))+1 = {idf_emb:.6f}")
print("\n'the' appears in almost all documents so (N+1)/(df+1) approaches 1, and log(1)=0.")
print("'embroidery' is rare so (N+1)/(df+1) is large, giving a high log value.")


IDF('the'):
  Docs containing 'the'        : 2992/10199
  IDF = log((10199+1)/(2992+1))+1 = 2.226111

IDF('embroidery'):
  Docs containing 'embroidery' : 0/10199
  IDF = log((10199+1)/(0+1))+1 = 10.230143

'the' appears in almost all documents so (N+1)/(df+1) approaches 1, and log(1)=0.
'embroidery' is rare so (N+1)/(df+1) is large, giving a high log value.


## (c) Rebuttal: Why TF-IDF over Raw Frequency?

In [9]:
rebuttal = (
    "1. Raw word frequency unfairly boosts common stop words like 'the' and 'is' "
    "that carry no meaning, making many irrelevant documents look similar to the query.\n\n"
    "2. TF-IDF downweights ubiquitous words through IDF, so only words that are both "
    "frequent in a specific document AND rare in the corpus get high scores — exactly "
    "the words that make a document distinctive.\n\n"
    "3. In practice, a search for 'wireless earbuds' using raw TF would rank documents "
    "with many generic words higher than earbuds-specific reviews; TF-IDF surfaces the "
    "correct documents by identifying words with real discriminating power."
)
print(rebuttal)


1. Raw word frequency unfairly boosts common stop words like 'the' and 'is' that carry no meaning, making many irrelevant documents look similar to the query.

2. TF-IDF downweights ubiquitous words through IDF, so only words that are both frequent in a specific document AND rare in the corpus get high scores — exactly the words that make a document distinctive.

3. In practice, a search for 'wireless earbuds' using raw TF would rank documents with many generic words higher than earbuds-specific reviews; TF-IDF surfaces the correct documents by identifying words with real discriminating power.


## Bonus: BM25 Scoring (k1=1.5, b=0.75)

In [10]:
def build_bm25_idf(token_series):
    N = len(token_series)
    df_counts = Counter()
    for tokens in token_series:
        for word in set(tokens):
            df_counts[word] += 1
    return {word: math.log((N - cnt + 0.5) / (cnt + 0.5) + 1)
            for word, cnt in df_counts.items()}

def compute_bm25_score(query_tokens, doc_tokens, idf_dict, avgdl, k1=1.5, b=0.75):
    dl = len(doc_tokens)
    score = 0.0
    for term in query_tokens:
        if term not in idf_dict:
            continue
        tf = doc_tokens.count(term)
        idf_val = idf_dict[term]
        score += idf_val * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl))
    return score

query_tokens = ["wireless", "earbuds", "battery", "life", "poor"]
avgdl = df_all["tokens"].apply(len).mean()
bm25_idf = build_bm25_idf(df_all["tokens"])

df_all["bm25_score"] = df_all["tokens"].apply(
    lambda toks: compute_bm25_score(query_tokens, toks, bm25_idf, avgdl)
)

top5_bm25 = df_all.nlargest(5, "bm25_score")[["review_id", "category", "bm25_score", "review_text"]]
print("Top-5 by BM25 | query:", query_tokens)
print(top5_bm25.to_string())
print("\nBM25 normalizes by document length so short reviews with one match are not")
print("over-rewarded, and very long reviews are not penalized as much as in raw TF-IDF.")


Top-5 by BM25 | query: ['wireless', 'earbuds', 'battery', 'life', 'poor']
     review_id     category  bm25_score                                                                          review_text
2617   R002567  Electronics    4.428893  Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!
6137   R006028  Electronics    4.428893  Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!
6180   R006070  Electronics    4.428893  Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!
7143   R007006  Electronics    4.428893  Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!
7920   R007768  Electronics    4.428893  Received a defective earbuds. Asked for replacement, still waiting after 2 weeks!!!

BM25 normalizes by document length so short reviews with one match are not
over-rewarded, and very long reviews are not penalized as much as in raw TF-IDF.
